In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    roc_curve
)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection")

print("Path to dataset files:", path)

In [ ]:
# from pathlib import Path
no_brain_cancer_data = []
yes_brain_cancer_data = []
no_brain_cancer_labels = []
yes_brain_cancer_labels = []

for child in Path.iterdir(Path(path + '/yes')):
  image = cv2.imread(child)
  resized = cv2.resize(image, (224, 224))
  yes_brain_cancer_data.append(resized)
  yes_brain_cancer_labels.append(1)

for child in Path.iterdir(Path(path + '/no')):
  image = cv2.imread(child)
  resized = cv2.resize(image, (224, 224))
  no_brain_cancer_data.append(resized)
  no_brain_cancer_labels.append(0)

In [ ]:
yes_brain_cancer_data[32]

In [ ]:
no_brain_cancer_labels

In [ ]:
X = yes_brain_cancer_data + no_brain_cancer_data
y = yes_brain_cancer_labels + no_brain_cancer_labels

In [ ]:
X = np.array(X)
y = np.array(y)

X.shape

In [ ]:
y.shape

In [ ]:
X.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42, stratify=y_test)

X_train.shape, X_val.shape, X_test.shape

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=X_train.shape[1:]),
    tf.keras.layers.Conv2D(filters=32, kernel_size=3, strides=2, padding='valid', kernel_initializer='he_normal',
                           kernel_regularizer=tf.keras.regularizers.L2(0.01)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=3, strides=2),

    tf.keras.layers.Conv2D(filters=32, kernel_size=3, strides=2, padding='valid', kernel_initializer='he_normal',
                           kernel_regularizer=tf.keras.regularizers.L2(0.01)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=3, strides=1),

    tf.keras.layers.Conv2D(filters=32, kernel_size=3, strides=2, padding='valid', kernel_initializer='he_normal',
                           kernel_regularizer=tf.keras.regularizers.L2(0.01)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=3, strides=1),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(units=64 ,kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Dense(units=64, kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Dense(units=1, activation='sigmoid')

])

model.summary()

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-03),
    metrics=['Accuracy']
)

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='checkpointbr.weights.h5',
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=True
)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='auto',
    patience=5,
    verbose=1,
    restore_best_weights=True
)


history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=100,
                    callbacks=[checkpoint_callback, early_stopping_callback])

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
y_test

In [ ]:
loss_ = history.history['loss']
val_loss_ = history.history['val_loss']
accuracy_ = history.history['Accuracy']
val_accuracy_ = history.history['val_Accuracy']


fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(14, 12), constrained_layout=True)


axs[0].plot(loss_)
axs[0].plot(val_loss_)
axs[0].set_title('Model Loss', fontsize=12)
axs[0].set_xlabel('Epochs', fontsize=12)
axs[0].set_ylabel('Losses', fontsize=12)
axs[0].legend(['train_loss', 'val_loss'])
axs[0].grid()


axs[1].plot(accuracy_)
axs[1].plot(val_accuracy_)
axs[1].set_title("Model Accuracy", fontsize=12)
axs[1].set_xlabel("Epochs", fontsize=12)
axs[1].set_ylabel('Accuracy', fontsize=12)
axs[1].legend(['train_accuracy','val_accuracy'])
axs[1].grid()


predictions = model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, (predictions >= 0.5).astype(int), ax=axs[2])
plt.show()



In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, predictions)

plt.plot(fpr, tpr, 'b', linewidth=2)
plt.plot([0, 1], [0, 1], 'k:')
plt.xlim(0, 1)
plt.ylim(0, 1)